# RAG 第 11 课：真实数据集 + 本地 LLM

这一课我们正式从教学 toy data 走向更真实的配置：

- 用 Hugging Face `datasets` 加载真实数据集
- 用本地 OpenAI 兼容接口拿 embedding
- 用本地 LLM 生成答案
- 在一个小型真实样本集上做最基本的检索与答案评估

这节课的目标不是做生产级 RAG，而是帮你打通：

`真实数据 -> 检索 -> 本地 LLM -> 评估`

补充一点背景：
- Hugging Face `transformers` 里确实有历史上的 `RAG` 模型家族
- 但今天更常见的工程路线，通常是：
  `datasets + retriever/index + LLM API`

所以这节课我们会走更贴近实际工程的这一条。

In [ ]:
# 先清理 GPU 缓存。
# 这份 notebook 本身尽量不直接占用 GPU，主要通过 LM Studio 的本地服务调用模型。
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 环境与服务配置

你已经启动了 LM Studio，本课直接使用 OpenAI 兼容接口：

- Base URL: `http://10.0.0.63:1234/v1`

这里我们会自动列出可用模型，并尽量自动挑一个：
- 一个 chat model
- 一个 embedding model

In [ ]:
import math
import re
from collections import Counter
from typing import List

import numpy as np
from datasets import load_dataset
from openai import OpenAI

BASE_URL = 'http://10.0.0.63:1234/v1'
API_KEY = 'lm-studio'

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
models = client.models.list().data
model_ids = [m.id for m in models]
print('available models:', model_ids)

embedding_model = next((m for m in model_ids if 'embedding' in m.lower() or 'embed' in m.lower()), None)
chat_model = next((m for m in model_ids if m != embedding_model), None)

print('embedding_model =', embedding_model)
print('chat_model      =', chat_model)

if embedding_model is None:
    raise RuntimeError('没有找到 embedding model，请先在 LM Studio 里加载一个 embedding 模型。')
if chat_model is None:
    raise RuntimeError('没有找到 chat model，请先在 LM Studio 里加载一个对话模型。')

## 2. 加载真实数据集

这里我们使用 Hugging Face 上非常经典的 `squad` 数据集。

为什么适合做这一课：
- 问题、上下文、参考答案都齐
- 数据量不小，但我们可以先切一小段做教学 demo
- 很适合演示“从上下文里回答问题”这件事

In [ ]:
# 为了让这节课跑得更快，我们只取 validation 的前一小段。
raw_ds = load_dataset('squad', split='validation[:120]')
print(raw_ds)
print('example question:', raw_ds[0]['question'])
print('example answer:', raw_ds[0]['answers']['text'][0])

## 3. 构建语料库与评测样本

`squad` 本身是一条条 `(question, context, answer)` 样本。
但 RAG 检索器更像是：

- 有一个文档库
- query 来了以后去检索文档

所以这里我们会：
1. 先把去重后的 `context` 作为文档库
2. 再把若干 `(question, gold_doc_id, answer)` 作为评测集

In [ ]:
context_to_doc_id = {}
documents = []
eval_rows = []

for item in raw_ds:
    context = item['context'].strip()
    if context not in context_to_doc_id:
        doc_id = len(documents)
        context_to_doc_id[context] = doc_id
        documents.append({'doc_id': doc_id, 'text': context, 'title': item['title']})
    else:
        doc_id = context_to_doc_id[context]

    answer_texts = item['answers']['text']
    if answer_texts:
        eval_rows.append({
            'question': item['question'].strip(),
            'gold_doc_id': doc_id,
            'reference_answer': answer_texts[0].strip(),
            'title': item['title'],
        })

print('num_documents =', len(documents))
print('num_eval_rows =', len(eval_rows))
print('first document title =', documents[0]['title'])
print('first eval row =', eval_rows[0])

## 4. Embedding：通过本地 OpenAI 兼容接口获取向量

这一步是这节课的关键升级点：

- 前几课我们用 toy embedding / toy retrieval
- 这节课我们改成真实 embedding model

而且这里 embedding 也是通过 LM Studio 服务提供，所以 notebook 本身不用直接吃 GPU。

In [ ]:
def get_embeddings(texts: List[str], batch_size: int = 16) -> np.ndarray:
    all_vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=embedding_model, input=batch)
        batch_vectors = [np.array(item.embedding, dtype=np.float32) for item in response.data]
        all_vectors.extend(batch_vectors)
        print(f'embedded {min(start + batch_size, len(texts))}/{len(texts)}')
    return np.vstack(all_vectors)


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return matrix / norms

In [ ]:
doc_texts = [doc['text'] for doc in documents]
doc_embeddings = get_embeddings(doc_texts, batch_size=16)
doc_embeddings = l2_normalize(doc_embeddings)
print('doc_embeddings shape =', doc_embeddings.shape)

## 5. 检索器

这里先用最基础的余弦相似度检索。
真正大规模生产环境里，通常会接向量库或 ANN 索引；但在这节课里，暴力检索更直观。

In [ ]:
def retrieve(question: str, top_k: int = 3):
    q_emb = get_embeddings([question], batch_size=1)
    q_emb = l2_normalize(q_emb)[0]
    scores = doc_embeddings @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            'doc_id': int(documents[idx]['doc_id']),
            'title': documents[idx]['title'],
            'score': float(scores[idx]),
            'text': documents[idx]['text'],
        })
    return results

## 6. 生成答案：通过本地 LLM

我们让模型只根据检索到的上下文回答，并尽量输出简短答案。

你问到 `64K` 上下文够不够，这节课里肯定是够的：
- 我们只放 `top-3` 的上下文
- 每条上下文也不算特别长

所以这个 demo 离 64K 还很远。

In [ ]:
def build_context_block(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, start=1):
        parts.append(f'[Document {i}] title: {doc["title"]}\n{doc["text"]}')
    return '\n\n'.join(parts)


def answer_with_llm(question: str, retrieved_docs, temperature: float = 0.0) -> str:
    context_block = build_context_block(retrieved_docs)
    system_prompt = (
        'You are a careful question-answering assistant. '
        'Answer only from the provided context. '
        'If the answer is not supported by the context, reply with NOT_FOUND. '
        'Keep the answer short, ideally a short phrase or one sentence.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Context:\n{context_block}\n\n'
        'Return only the answer.'
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

## 7. 先跑一条完整样本

这一格很重要，因为你会第一次看到：

`真实 question -> 真实检索 -> 本地 LLM answer`

如果这一步跑通，后面的批量评估就只是把它重复很多次。

In [ ]:
sample = eval_rows[0]
retrieved = retrieve(sample['question'], top_k=3)
pred_answer = answer_with_llm(sample['question'], retrieved)

print('question:')
print(sample['question'])
print('\ngold_doc_id:', sample['gold_doc_id'])
print('reference_answer:', sample['reference_answer'])
print('\nretrieved top-3:')
for row in retrieved:
    print(f"doc_id={row['doc_id']} | score={row['score']:.4f} | title={row['title']}")
    print(row['text'][:240].replace('\n', ' '), '...')
    print()
print('predicted answer:')
print(pred_answer)

## 8. 评估函数

为了保持这节课重点清楚，我们先看两层最基础评估：

- 检索：`RetrievalHit@k`
- 答案：`Exact Match` + `Token F1`

这不是完整 QA benchmark，但足够做第一版真实验证。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(tokenize(text))


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 9. 在一个小真实样本集上批量评估

为了控制调用成本，这里我们只先跑前 5 条。
你后面熟悉之后，可以把数量加到 `10`、`20`、`50`。

In [ ]:
eval_subset = eval_rows[:5]
results = []

for i, row in enumerate(eval_subset, start=1):
    print(f'processing {i}/{len(eval_subset)}: {row["question"]}')
    retrieved = retrieve(row['question'], top_k=3)
    pred_answer = answer_with_llm(row['question'], retrieved)

    retrieved_ids = [doc['doc_id'] for doc in retrieved]
    retrieval_hit = 1.0 if row['gold_doc_id'] in retrieved_ids else 0.0
    em = exact_match(pred_answer, row['reference_answer'])
    f1 = token_f1(pred_answer, row['reference_answer'])

    results.append({
        'question': row['question'],
        'gold_doc_id': row['gold_doc_id'],
        'retrieved_ids': retrieved_ids,
        'reference_answer': row['reference_answer'],
        'pred_answer': pred_answer,
        'retrieval_hit@3': retrieval_hit,
        'exact_match': em,
        'token_f1': f1,
    })

summary = {
    'RetrievalHit@3': float(np.mean([x['retrieval_hit@3'] for x in results])),
    'ExactMatch': float(np.mean([x['exact_match'] for x in results])),
    'TokenF1': float(np.mean([x['token_f1'] for x in results])),
}
print('\nsummary =', summary)

## 10. 看逐条结果

真实调试时，这一格往往比平均分还重要。

In [ ]:
for row in results:
    print('question:', row['question'])
    print('gold_doc_id:', row['gold_doc_id'])
    print('retrieved_ids:', row['retrieved_ids'])
    print('reference_answer:', row['reference_answer'])
    print('pred_answer:', row['pred_answer'])
    print('retrieval_hit@3:', row['retrieval_hit@3'])
    print('exact_match:', row['exact_match'], '| token_f1:', round(row['token_f1'], 3))
    print('-' * 100)

## 11. 这一课你要建立的工程直觉

到这里，你其实已经从“教学版 RAG”走到了“真实版最小可运行 RAG”。

它已经具备真实系统的几个关键特征：
- 真实数据集
- 真实 embedding model
- 真实 LLM
- 真实的 retrieval / answer evaluation

当然它还很简化：
- 没有向量数据库
- 没有 chunking 策略优化
- 没有 reranker 模型
- 没有更严格的 answer judge

但作为你进入下一阶段的桥，这已经很够了。

## 12. 本课小结

这节课你要带走的核心是：

1. Hugging Face 的 `datasets` 很适合快速接入真实 RAG 数据。
2. 你完全可以用本地 OpenAI 兼容接口来做 embedding + generation。
3. 在 64K 上下文下，这种 `top-k` 小规模 RAG demo 完全够用。
4. 真正的升级方向，下一步会是：
   - 更大的数据集
   - 更好的 chunking
   - reranker
   - 更严格的评测

下一课最自然的衔接就是：
**把这套真实版 RAG 再升级成带 reranking 的版本。**